# Object-Centric Scene Representation (nuScenes)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hombosh-dev/SkyNet/blob/feat/object-centric-repr/notebooks/ObjectCentric_Representation.ipynb)

**Goal.** Turn every annotated agent of a nuScenes scene into a feature vector, check that no object is lost, look at the result, and measure the object-level baselines that an object-centric world model has to beat. This is the object-centric counterpart of `BEVFusion_Encoder.ipynb`.

**Input.** nuScenes `v1.0-mini` (10 scenes). Only the JSON tables are required, plus the keyframe LiDAR for plot backgrounds; the notebook finds an existing copy (e.g. from the BEV notebook) or streams the archive and keeps only those files (~40 MB of tables, ~600 MB of LiDAR).

**Self-contained.** No other file from the repository is needed; it runs as is on Colab or Kaggle (Kaggle: Internet on, or nuScenes mini attached as an input). The `# [library]` cells are copies of `src/skynet/`, and `tests/test_object_centric_notebook.py` keeps them identical.

**Representation.** One scene → `features [T, N_agents, 11]` + presence `mask [T, N_agents]`. The agent axis is the tracking id (one nuScenes instance per index). Features: position `x, y, z`, size `length, width, height`, heading `sin/cos yaw`, velocity `vx, vy` (backward difference, never reads later frames), coarse class.

**Conclusion.** *Write after Run All, from the tables below.*

In [ ]:
# Environment, seeds, output folder
import glob
import io
import json
import math
import os
import random
import sys
import urllib.request
from dataclasses import dataclass
from pathlib import Path

import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Image as IPImage, display
from PIL import Image

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

IS_KAGGLE = os.path.exists("/kaggle")
IS_COLAB = "google.colab" in sys.modules
WORK_DIR = "/kaggle/working" if IS_KAGGLE else ("/content" if IS_COLAB else os.path.abspath("."))
OUT_DIR = os.path.join(WORK_DIR, "object_centric_outputs")
os.makedirs(OUT_DIR, exist_ok=True)
print(f"Python {sys.version.split()[0]} | PyTorch {torch.__version__} | outputs: {OUT_DIR}")

In [ ]:
# [library] Find or download nuScenes — same code as src/skynet/data/download.py
"""Find nuScenes on disk, or download the parts of v1.0-mini this project needs.

The full mini archive is ~4 GB, mostly camera images and LiDAR sweeps. The archive
is streamed and only the requested members are written to disk: the JSON tables
(~40 MB, enough for the object-centric representation and the ego baselines) and,
optionally, the keyframe LiDAR point clouds (~600 MB, used for plots). Nothing else
is stored, so no 4 GB file is left behind.
"""


import os
import shutil
import tarfile
import urllib.request
from pathlib import Path

MINI_URL = "https://www.nuscenes.org/data/v1.0-mini.tgz"
ENV_VAR = "NUSCENES_ROOT"
COMMON_LOCATIONS = (
    "data/nuscenes",
    "data/sets/nuscenes",
    "/content/data/sets/nuscenes",
    "/kaggle/working/data/sets/nuscenes",
    "~/data/sets/nuscenes",
)


def has_tables(root: str | Path, version: str = "v1.0-mini") -> bool:
    return (Path(root).expanduser() / version / "sample.json").exists()


def find_nuscenes(candidates=(), version: str = "v1.0-mini") -> Path | None:
    """First folder holding ``<version>/sample.json``: the given candidates, then
    ``$NUSCENES_ROOT``, then common Colab / Kaggle / local locations."""
    roots = [c for c in candidates if c] + [os.environ.get(ENV_VAR, "")] + list(COMMON_LOCATIONS)
    for root in roots:
        if root and has_tables(root, version):
            return Path(root).expanduser()
    return None


def _wanted(name: str, version: str, with_lidar: bool) -> str | None:
    """Path relative to the dataset root if the archive member is needed, else None."""
    parts = Path(name).parts
    for i, part in enumerate(parts):
        if part == version and name.endswith(".json"):
            return str(Path(*parts[i:]))
        if with_lidar and part == "samples" and parts[i + 1 : i + 2] == ("LIDAR_TOP",):
            return str(Path(*parts[i:]))
    return None


def extract_needed(
    fileobj, root: str | Path, version: str = "v1.0-mini", with_lidar: bool = False
) -> int:
    """Stream a .tgz and write only the JSON tables (and keyframe LiDAR) under ``root``.

    Returns the number of files written. Works on a non-seekable stream.
    """
    root = Path(root)
    written = 0
    with tarfile.open(fileobj=fileobj, mode="r|gz") as archive:
        for member in archive:
            rel = _wanted(member.name, version, with_lidar) if member.isfile() else None
            if rel is None:
                continue
            target = root / rel
            target.parent.mkdir(parents=True, exist_ok=True)
            with archive.extractfile(member) as src, target.open("wb") as dst:
                shutil.copyfileobj(src, dst)
            written += 1
    return written


def download_mini(root: str | Path, with_lidar: bool = False, url: str = MINI_URL) -> Path:
    """Download nuScenes v1.0-mini into ``root`` (tables, optionally keyframe LiDAR).

    By downloading you accept the nuScenes terms of use (CC BY-NC-SA 4.0).
    """
    root = Path(root).expanduser()
    print(f"Streaming {url} (~4 GB) and keeping only what is needed ...")
    with urllib.request.urlopen(url) as response:
        written = extract_needed(response, root, with_lidar=with_lidar)
    if not has_tables(root):
        raise RuntimeError(f"download finished but no tables were found in {root}")
    print(f"Wrote {written} files to {root}")
    return root


def find_or_download(
    preferred: str | Path | None = None,
    download_to: str | Path = "data/nuscenes",
    version: str = "v1.0-mini",
    with_lidar: bool = False,
) -> Path:
    """Return a nuScenes root, downloading mini into ``download_to`` if none is found."""
    found = find_nuscenes([str(preferred)] if preferred else [], version)
    if found is not None:
        return found
    if version != "v1.0-mini":
        raise FileNotFoundError(f"nuScenes {version} not found; set ${ENV_VAR} or --data-root")
    return download_mini(download_to, with_lidar=with_lidar)

In [ ]:
# nuScenes v1.0-mini: reuse a copy if there is one (BEV notebook, Kaggle input, Drive), else stream it.
VERSION = "v1.0-mini"
WITH_LIDAR = True  # keyframe LiDAR for plot backgrounds; set False to keep only the JSON tables

kaggle_inputs = [str(Path(p).parent.parent) for p in glob.glob(f"/kaggle/input/**/{VERSION}/sample.json", recursive=True)]
DATA_ROOT = find_nuscenes(kaggle_inputs + [os.path.join(WORK_DIR, "data/sets/nuscenes")], VERSION)
if DATA_ROOT is None:
    DATA_ROOT = Path(WORK_DIR) / "data/sets/nuscenes"
    local_archives = [p for p in ["/content/drive/MyDrive/v1.0-mini.tgz", "/content/v1.0-mini.tgz", "/kaggle/working/v1.0-mini.tgz"] if os.path.exists(p)]
    if local_archives:
        with open(local_archives[0], "rb") as f:
            extract_needed(f, DATA_ROOT, VERSION, with_lidar=WITH_LIDAR)
    else:
        download_mini(DATA_ROOT, with_lidar=WITH_LIDAR)
HAS_LIDAR = any((Path(DATA_ROOT) / "samples" / "LIDAR_TOP").glob("*.bin"))
print("nuScenes tables:", Path(DATA_ROOT) / VERSION, "| keyframe LiDAR available:", HAS_LIDAR)

In [ ]:
# [library] nuScenes tables (keyframes only, no devkit) — same code as src/skynet/data/nuscenes.py
"""Minimal, dependency-free reader of the nuScenes JSON tables (no nuscenes-devkit).

Only keyframes are indexed: ``sample_data.json`` also contains sweeps (LiDAR at 20 Hz,
cameras at 12 Hz) that carry a keyframe's ``sample_token``, so sensor records are
filtered on ``is_key_frame``. Scene frames are ordered by following
``scene.first_sample_token -> next``, not by the order of ``sample.json``.
"""


import json
import math
from pathlib import Path

import numpy as np

# Official mini splits (nuscenes-devkit, nuscenes/utils/splits.py).
MINI_SPLITS = {
    "mini_train": [
        "scene-0061",
        "scene-0553",
        "scene-0655",
        "scene-0757",
        "scene-0796",
        "scene-1077",
        "scene-1094",
        "scene-1100",
    ],
    "mini_val": ["scene-0103", "scene-0916"],
}


def quaternion_to_matrix(q) -> np.ndarray:
    """3x3 rotation matrix of a quaternion ``[w, x, y, z]``."""
    w, x, y, z = np.asarray(q, dtype=np.float64) / np.linalg.norm(q)
    return np.array(
        [
            [1 - 2 * (y * y + z * z), 2 * (x * y - z * w), 2 * (x * z + y * w)],
            [2 * (x * y + z * w), 1 - 2 * (x * x + z * z), 2 * (y * z - x * w)],
            [2 * (x * z - y * w), 2 * (y * z + x * w), 1 - 2 * (x * x + y * y)],
        ]
    )


def yaw_of(rotation: np.ndarray) -> float:
    """Heading (rotation about z) of a 3x3 rotation matrix, in radians."""
    return math.atan2(rotation[1, 0], rotation[0, 0])


class NuScenesTables:
    """Indexed nuScenes metadata.

    Parameters
    ----------
    dataroot : str or Path
        Folder that contains the ``<version>/`` folder with the JSON tables.
    version : str
        ``v1.0-mini`` or ``v1.0-trainval``.
    """

    def __init__(self, dataroot: str | Path, version: str = "v1.0-mini"):
        meta = Path(dataroot) / version
        if not (meta / "sample.json").exists():
            raise FileNotFoundError(f"nuScenes tables not found in {meta}")

        def load(name: str) -> list[dict]:
            with (meta / f"{name}.json").open(encoding="utf-8") as f:
                return json.load(f)

        self.scenes = load("scene")
        self.scene_by_name = {s["name"]: s for s in self.scenes}
        self.sample = {r["token"]: r for r in load("sample")}
        self.ego_pose_records = {r["token"]: r for r in load("ego_pose")}
        calib = {r["token"]: r for r in load("calibrated_sensor")}
        channel_of = {r["token"]: r["channel"] for r in load("sensor")}
        categories = {r["token"]: r["name"] for r in load("category")}
        self.category_of_instance = {
            r["token"]: categories[r["category_token"]] for r in load("instance")
        }
        self.keyframe: dict[tuple[str, str], dict] = {}
        for rec in load("sample_data"):
            if rec["is_key_frame"]:
                channel = channel_of[calib[rec["calibrated_sensor_token"]]["sensor_token"]]
                self.keyframe[(rec["sample_token"], channel)] = rec
        self.annotation: dict[str, dict] = {}
        self.annotations_of: dict[str, list[dict]] = {}
        for ann in load("sample_annotation"):
            self.annotation[ann["token"]] = ann
            self.annotations_of.setdefault(ann["sample_token"], []).append(ann)

    def split_scene_names(self, split: str) -> list[str]:
        """Scene names of ``mini_train``, ``mini_val`` or ``all``."""
        if split == "all":
            return [s["name"] for s in self.scenes]
        if split not in MINI_SPLITS:
            raise ValueError(f"unknown split {split!r}; use mini_train, mini_val or all")
        return list(MINI_SPLITS[split])

    def scene_sample_tokens(self, scene_name: str) -> list[str]:
        """Keyframe sample tokens of a scene in chronological order."""
        if scene_name not in self.scene_by_name:
            raise KeyError(f"scene {scene_name!r} is not in this nuScenes version")
        tokens, token = [], self.scene_by_name[scene_name]["first_sample_token"]
        while token:
            tokens.append(token)
            token = self.sample[token]["next"]
        return tokens

    def timestamp(self, sample_token: str) -> float:
        """Sample timestamp in seconds."""
        return self.sample[sample_token]["timestamp"] * 1e-6

    def ego_pose(self, sample_token: str) -> tuple[np.ndarray, np.ndarray]:
        """Ego -> global rotation (3, 3) and translation (3,) at the LiDAR keyframe."""
        rec = self.keyframe[(sample_token, "LIDAR_TOP")]
        pose = self.ego_pose_records[rec["ego_pose_token"]]
        return quaternion_to_matrix(pose["rotation"]), np.asarray(pose["translation"], float)

In [ ]:
# [library] Object-centric scene representation — same code as src/skynet/models/object_centric/scene_repr.py
"""Object-centric scene representation: every annotated agent as a feature vector.

One nuScenes scene becomes a dense tensor ``features`` of shape ``(T, N, F)``:
``T`` keyframes, ``N`` agents (one per nuScenes instance, i.e. the tracking id is the
index on the agent axis) and ``F = len(FEATURES)`` features. ``mask[t, n]`` is True
where agent ``n`` is annotated at keyframe ``t``; elsewhere the row is zero. For one
frame, :meth:`ObjectScene.frame` returns the ``(N_agents, F)`` matrix of the agents
present. Every annotation of the scene is kept (no category, distance or LiDAR-point
filter), so ``mask.sum()`` equals the scene's number of annotations.

Velocity is a *backward* difference to the instance's previous annotation, so the
representation of frame t uses no information from later frames and is safe as a
world-model input. Agents seen for the first time have zero velocity and
``velocity_valid`` False. (nuscenes-devkit's ``box_velocity`` uses a centred
difference, which reads the next frame.)
"""


import math
from dataclasses import dataclass

import numpy as np
import torch


FEATURES = (
    "x",
    "y",
    "z",
    "length",
    "width",
    "height",
    "yaw_sin",
    "yaw_cos",
    "vx",
    "vy",
    "class_id",
)
CLASSES = ("vehicle", "pedestrian", "two_wheeler", "obstacle", "animal")
FRAMES = ("ego", "global")


def class_id(category: str) -> int:
    """Coarse class of a nuScenes category name, as an index into ``CLASSES``."""
    if category.startswith(("vehicle.bicycle", "vehicle.motorcycle")):
        return CLASSES.index("two_wheeler")
    if category.startswith("vehicle."):
        return CLASSES.index("vehicle")
    if category.startswith("human.pedestrian"):
        return CLASSES.index("pedestrian")
    if category.startswith("animal"):
        return CLASSES.index("animal")
    return CLASSES.index("obstacle")  # movable_object.*, static_object.*


@dataclass
class ObjectScene:
    """Object-centric tensors of one scene (see the module docstring)."""

    scene_name: str
    frame: str  # "ego": each keyframe in its own ego frame; "global": world frame
    sample_tokens: list[str]  # (T,) chronological keyframes
    timestamps: torch.Tensor  # (T,) seconds since the first keyframe
    track_ids: list[str]  # (N,) nuScenes instance tokens; index = tracking id
    categories: list[str]  # (N,) nuScenes category names
    features: torch.Tensor  # (T, N, F) float32
    mask: torch.Tensor  # (T, N) bool
    velocity_valid: torch.Tensor  # (T, N) bool

    @property
    def num_annotations(self) -> int:
        return int(self.mask.sum())

    def frame_features(self, t: int) -> torch.Tensor:
        """``(N_agents, F)`` features of the agents present at keyframe ``t``."""
        return self.features[t][self.mask[t]]

    def frame_track_ids(self, t: int) -> list[str]:
        """Instance tokens matching the rows of :meth:`frame_features`."""
        return [self.track_ids[n] for n in torch.nonzero(self.mask[t]).flatten().tolist()]


def build_object_scene(tables: NuScenesTables, scene_name: str, frame: str = "ego") -> ObjectScene:
    """Convert all annotations of one scene into an :class:`ObjectScene`.

    Parameters
    ----------
    tables : NuScenesTables
    scene_name : str
        e.g. ``"scene-0061"``.
    frame : str
        ``"ego"`` (default): positions, yaw and velocity of keyframe t are expressed
        in the ego frame of keyframe t (x forward, y left). ``"global"``: world frame,
        convenient for forecasting metrics across frames.
    """
    if frame not in FRAMES:
        raise ValueError(f"frame must be one of {FRAMES}, got {frame!r}")
    tokens = tables.scene_sample_tokens(scene_name)
    stamps = [tables.timestamp(t) for t in tokens]

    track_ids: list[str] = []
    index: dict[str, int] = {}
    for token in tokens:  # agents ordered by first appearance, then by instance token
        for inst in sorted(a["instance_token"] for a in tables.annotations_of.get(token, [])):
            if inst not in index:
                index[inst] = len(track_ids)
                track_ids.append(inst)

    T, N = len(tokens), len(track_ids)
    features = np.zeros((T, N, len(FEATURES)), dtype=np.float32)
    mask = np.zeros((T, N), dtype=bool)
    vel_valid = np.zeros((T, N), dtype=bool)

    for t, token in enumerate(tokens):
        if frame == "ego":
            rot, trans = tables.ego_pose(token)
        else:
            rot, trans = np.eye(3), np.zeros(3)
        for ann in tables.annotations_of.get(token, []):
            n = index[ann["instance_token"]]
            if mask[t, n]:
                raise ValueError(f"instance {ann['instance_token']} annotated twice at {token}")
            pos = rot.T @ (np.asarray(ann["translation"], float) - trans)
            yaw = yaw_of(rot.T @ quaternion_to_matrix(ann["rotation"]))
            vel = np.zeros(2)
            prev = tables.annotation.get(ann["prev"]) if ann["prev"] else None
            if prev is not None:
                dt = tables.timestamp(ann["sample_token"]) - tables.timestamp(prev["sample_token"])
                v_global = (np.asarray(ann["translation"]) - np.asarray(prev["translation"])) / dt
                vel = (rot.T @ v_global)[:2]
                vel_valid[t, n] = True
            width, length, height = ann["size"]  # nuScenes order is [w, l, h]
            features[t, n] = [
                *pos,
                length,
                width,
                height,
                math.sin(yaw),
                math.cos(yaw),
                *vel,
                class_id(tables.category_of_instance[ann["instance_token"]]),
            ]
            mask[t, n] = True

    return ObjectScene(
        scene_name=scene_name,
        frame=frame,
        sample_tokens=tokens,
        timestamps=torch.tensor([s - stamps[0] for s in stamps], dtype=torch.float64),
        track_ids=track_ids,
        categories=[tables.category_of_instance[i] for i in track_ids],
        features=torch.from_numpy(features),
        mask=torch.from_numpy(mask),
        velocity_valid=torch.from_numpy(vel_valid),
    )

In [ ]:
# [library] Object forecasting metrics: ADE / FDE / miss rate — same code as src/skynet/eval/object_metrics.py
"""Forecasting metrics for the object-centric representation: ADE, FDE, miss rate.

These are the standard agent-forecasting metrics; they complement the shared metrics
of the benchmark (L2 / collision rate for the ego, Chamfer Distance on point clouds).
Horizons follow the planning metrics: nuScenes keyframes are 2 Hz, so 1 s, 2 s and
3 s are steps 2, 4 and 6.
"""


import torch

HORIZON_STEPS = {"1s": 2, "2s": 4, "3s": 6}


def displacement_errors(
    pred: torch.Tensor,
    gt: torch.Tensor,
    valid: torch.Tensor | None = None,
    miss_threshold_m: float = 2.0,
) -> dict[str, dict[str, float]]:
    """ADE, FDE and miss rate of agent trajectories, per horizon.

    Parameters
    ----------
    pred, gt : torch.Tensor
        Agent positions of shape (A, T, 2), T >= 6, in metres (any common frame).
    valid : torch.Tensor, optional
        Bool (A, T): False where the agent is not annotated at that step. Masked
        steps are ignored; a horizon with no valid step gives NaN.
    miss_threshold_m : float
        An agent is missed at horizon h if its error at step 2h exceeds this value.

    Returns
    -------
    dict
        ``{"ade": {...}, "fde": {...}, "miss_rate": {...}}`` with keys "1s", "2s",
        "3s". ADE averages the error over all valid (agent, step) pairs up to the
        horizon; FDE and miss rate use the agents valid at the horizon step.
    """
    if pred.shape != gt.shape or pred.ndim != 3 or pred.shape[-1] != 2:
        raise ValueError(
            f"expected matching (A, T, 2) tensors, got {tuple(pred.shape)} / {tuple(gt.shape)}"
        )
    if pred.shape[1] < max(HORIZON_STEPS.values()):
        raise ValueError(f"need at least {max(HORIZON_STEPS.values())} steps")
    if valid is None:
        valid = torch.ones(pred.shape[:2], dtype=torch.bool)
    if tuple(valid.shape) != tuple(pred.shape[:2]):
        raise ValueError(f"valid must have shape {tuple(pred.shape[:2])}, got {tuple(valid.shape)}")

    dist = torch.linalg.norm(pred.double() - gt.double(), dim=-1)
    out: dict[str, dict[str, float]] = {"ade": {}, "fde": {}, "miss_rate": {}}
    nan = float("nan")
    for name, n in HORIZON_STEPS.items():
        upto = valid[:, :n]
        at = valid[:, n - 1]
        out["ade"][name] = float(dist[:, :n][upto].mean()) if upto.any() else nan
        out["fde"][name] = float(dist[:, n - 1][at].mean()) if at.any() else nan
        missed = (dist[:, n - 1][at] > miss_threshold_m).double()
        out["miss_rate"][name] = float(missed.mean()) if at.any() else nan
    return out

In [ ]:
# [library] Object baselines: persistence and constant velocity — same code as src/skynet/baselines/object_centric.py
"""Baselines on the object-centric representation and their evaluation windows.

* ``persistence``: every agent stays where it was last seen.
* ``constant_velocity``: every agent keeps its last (backward-difference) velocity.

Both use only the current keyframe, so they are the minimum any learned
object-centric world model has to beat.
"""


import torch


X, Y, VX, VY = (FEATURES.index(k) for k in ("x", "y", "vx", "vy"))


def forecast_windows(scene: ObjectScene, num_future: int = 6):
    """Yield (t0, positions_t0 (A, 2), velocities_t0 (A, 2), future_gt (A, T, 2),
    future_valid (A, T), dt (T,)) for every keyframe with ``num_future`` successors.

    ``scene`` must use ``frame="global"`` so that positions of different keyframes
    are comparable. Agents are those annotated at t0.
    """
    if scene.frame != "global":
        raise ValueError("forecast windows need an ObjectScene built with frame='global'")
    feats, mask = scene.features, scene.mask
    for t0 in range(feats.shape[0] - num_future):
        agents = mask[t0]
        if not agents.any():
            continue
        steps = slice(t0 + 1, t0 + 1 + num_future)
        yield (
            t0,
            feats[t0, agents][:, [X, Y]].double(),
            feats[t0, agents][:, [VX, VY]].double(),
            feats[steps][:, agents][..., [X, Y]].transpose(0, 1).double(),
            mask[steps][:, agents].transpose(0, 1),
            scene.timestamps[steps] - scene.timestamps[t0],
        )


def persistence(pos: torch.Tensor, vel: torch.Tensor, dt: torch.Tensor) -> torch.Tensor:
    """(A, T, 2): the t0 position repeated."""
    return pos[:, None, :].expand(-1, len(dt), -1).clone()


def constant_velocity(pos: torch.Tensor, vel: torch.Tensor, dt: torch.Tensor) -> torch.Tensor:
    """(A, T, 2): straight-line motion with the t0 velocity."""
    return pos[:, None, :] + dt[None, :, None] * vel[:, None, :]


BASELINES = {"persistence": persistence, "constant_velocity": constant_velocity}

In [ ]:
# [library] Planning metrics: ego L2 and collision rate — same code as src/skynet/eval/planning_metrics.py
"""Planning metrics: L2 error and collision rate at 1s, 2s, 3s (open-loop)"""


import math

import torch

# nuScenes keyframes are 2 Hz, so a horizon in seconds is 2 * seconds steps.
PLANNING_HORIZON_STEPS = {"1s": 2, "2s": 4, "3s": 6}

# Ego vehicle footprint in metres (nuScenes Renault Zoe, as used by UniAD/VAD).
EGO_LENGTH_M = 4.084
EGO_WIDTH_M = 1.85


def _with_avg(values: dict[str, float]) -> dict[str, float]:
    """Add the 'avg' entry: the plain mean of the per-horizon values."""
    out = dict(values)
    out["avg"] = sum(values.values()) / len(values)
    return out


def l2_error(pred: torch.Tensor, gt: torch.Tensor, cumulative: bool = False) -> dict[str, float]:
    """L2 distance between predicted and ground-truth ego positions, in metres.

    Parameters
    ----------
    pred, gt : torch.Tensor
        Ego x/y positions of shape (T, 2) for one scene or (B, T, 2) for a batch,
        with T >= 6 (2 Hz, so 6 steps = 3 s).
    cumulative : bool
        False: the value at horizon h is the error at step 2*h only.
        True: the value at horizon h is the mean error over all steps up to 2*h.
        With a batch, errors are averaged over scenes.

    Returns
    -------
    dict
        Keys "1s", "2s", "3s" and "avg" (mean of the three horizons).
    """
    if pred.shape != gt.shape:
        raise ValueError(f"shape mismatch: {tuple(pred.shape)} vs {tuple(gt.shape)}")
    if pred.ndim not in (2, 3) or pred.shape[-1] != 2:
        raise ValueError(f"expected (T, 2) or (B, T, 2), got {tuple(pred.shape)}")
    if pred.shape[-2] < max(PLANNING_HORIZON_STEPS.values()):
        raise ValueError(f"need at least {max(PLANNING_HORIZON_STEPS.values())} steps")

    dist = torch.linalg.norm(pred - gt, dim=-1)  # (T,) or (B, T)
    per_step = dist.reshape(-1, dist.shape[-1]).mean(dim=0)  # mean over scenes
    values = {}
    for name, n in PLANNING_HORIZON_STEPS.items():
        values[name] = float(per_step[:n].mean() if cumulative else per_step[n - 1])
    return _with_avg(values)


def _corners(cx: float, cy: float, length: float, width: float, yaw: float):
    """Corners of a rotated rectangle, as four (x, y) tuples in order."""
    c, s = math.cos(yaw), math.sin(yaw)
    hl, hw = length / 2, width / 2
    return [
        (cx + c * dx - s * dy, cy + s * dx + c * dy)
        for dx, dy in ((hl, hw), (hl, -hw), (-hl, -hw), (-hl, hw))
    ]


def _rects_overlap(a, b) -> bool:
    """Separating-axis test for two rectangles; touching counts as overlap."""
    for poly in (a, b):
        for i in range(4):
            x1, y1 = poly[i]
            x2, y2 = poly[(i + 1) % 4]
            nx, ny = -(y2 - y1), x2 - x1  # normal to this edge
            proj_a = [nx * x + ny * y for x, y in a]
            proj_b = [nx * x + ny * y for x, y in b]
            if max(proj_a) < min(proj_b) or max(proj_b) < min(proj_a):
                return False  # found a separating axis
    return True


def collision_rate(
    pred_traj: torch.Tensor,
    gt_boxes: list[list[torch.Tensor]],
    ego_size: tuple[float, float] = (EGO_LENGTH_M, EGO_WIDTH_M),
    cumulative: bool = False,
) -> dict[str, float]:
    """Fraction of scenes where the predicted ego footprint hits a ground-truth box.

    Parameters
    ----------
    pred_traj : torch.Tensor
        Predicted ego poses (x, y, yaw) of shape (B, T, 3), T >= 6.
    gt_boxes : list of list of torch.Tensor
        ``gt_boxes[b][t]`` is a (K, 5) tensor of objects at step t of scene b, each
        row (cx, cy, length, width, yaw); K may be 0.
    ego_size : tuple
        Ego (length, width) in metres.
    cumulative : bool
        False: a scene collides at horizon h if the ego hits something at step 2*h.
        True: if it hits something at any step up to 2*h.

    Returns
    -------
    dict
        Keys "1s", "2s", "3s" and "avg"; values are fractions in [0, 1].
    """
    if pred_traj.ndim != 3 or pred_traj.shape[-1] != 3:
        raise ValueError(f"expected (B, T, 3), got {tuple(pred_traj.shape)}")
    batch, steps = pred_traj.shape[0], max(PLANNING_HORIZON_STEPS.values())
    if pred_traj.shape[1] < steps:
        raise ValueError(f"need at least {steps} steps")
    if len(gt_boxes) != batch:
        raise ValueError("gt_boxes must have one entry per scene")

    hits = [[False] * steps for _ in range(batch)]
    for b in range(batch):
        for t in range(steps):
            x, y, yaw = pred_traj[b, t].tolist()
            ego = _corners(x, y, ego_size[0], ego_size[1], yaw)
            hits[b][t] = any(_rects_overlap(ego, _corners(*box)) for box in gt_boxes[b][t].tolist())

    values = {}
    for name, n in PLANNING_HORIZON_STEPS.items():
        flags = [any(h[:n]) if cumulative else h[n - 1] for h in hits]
        values[name] = sum(flags) / batch
    return _with_avg(values)

In [ ]:
# [library] Ego baselines that read only the past — same code as src/skynet/baselines/ego.py
"""Ego-trajectory baselines that read only the past.

* ``zero_motion``: the ego stays at the origin of its current frame.
* ``constant_velocity``: the ego keeps the velocity it had between the previous
  keyframe and now.

The constant-velocity baseline of the midterm notebook took its velocity from the
first *future* waypoint (``gt[0] / 0.5``), i.e. from the ground truth it is scored
against; this version is the replacement used for reported numbers.
"""


import numpy as np
import torch


def ego_windows(tables: NuScenesTables, scene_name: str, num_future: int = 6):
    """(sample_token, prev_xy (2,), dt_prev, future_xy (T, 2), future_dt (T,)) for
    every keyframe with a previous keyframe and ``num_future`` successors.

    Positions are in the ego frame of the current keyframe (x forward, y left).
    """
    tokens = tables.scene_sample_tokens(scene_name)
    out = []
    for i in range(1, len(tokens) - num_future):
        rot, trans = tables.ego_pose(tokens[i])

        def local(token):
            return (rot.T @ (tables.ego_pose(token)[1] - trans))[:2]

        now = tables.timestamp(tokens[i])
        future = tokens[i + 1 : i + 1 + num_future]
        out.append(
            (
                tokens[i],
                local(tokens[i - 1]),
                now - tables.timestamp(tokens[i - 1]),
                np.stack([local(t) for t in future]),
                np.array([tables.timestamp(t) - now for t in future]),
            )
        )
    return out


def ego_zero_motion(prev_xy: np.ndarray, dt_prev: float, future_dt: np.ndarray) -> np.ndarray:
    return np.zeros((len(future_dt), 2))


def ego_constant_velocity(prev_xy: np.ndarray, dt_prev: float, future_dt: np.ndarray) -> np.ndarray:
    velocity = -np.asarray(prev_xy) / dt_prev  # previous keyframe -> now
    return future_dt[:, None] * velocity[None, :]


EGO_BASELINES = {"zero_motion": ego_zero_motion, "constant_velocity": ego_constant_velocity}


def evaluate_ego_baselines(tables: NuScenesTables, scene_names: list[str], num_future: int = 6):
    """L2 (planning_metrics.l2_error) of each ego baseline over all windows of the scenes."""

    windows = [w for name in scene_names for w in ego_windows(tables, name, num_future)]
    gt = torch.tensor(np.stack([w[3] for w in windows]))
    out = {"num_windows": len(windows)}
    for name, fn in EGO_BASELINES.items():
        pred = torch.tensor(np.stack([fn(w[1], w[2], w[4]) for w in windows]))
        out[name] = l2_error(pred, gt)
    return out

In [ ]:
tables = NuScenesTables(DATA_ROOT, VERSION)
SCENE = "scene-0061"  # the scene the BEV notebook benchmarks on
scene = build_object_scene(tables, SCENE, frame="ego")
expected = sum(len(tables.annotations_of.get(t, [])) for t in scene.sample_tokens)

print(f"{SCENE}: features {tuple(scene.features.shape)}  (keyframes, agents, features)")
print(f"mask {tuple(scene.mask.shape)}, annotations in scene: {expected}, in tensor: {scene.num_annotations}")
assert scene.num_annotations == expected, "object loss!"
print("No object loss: every annotation is in the tensor.")
print("Features:", ", ".join(FEATURES))

t = 0
frame0 = pd.DataFrame(scene.frame_features(t).numpy(), columns=FEATURES)
frame0.insert(0, "track_id", [tid[:8] for tid in scene.frame_track_ids(t)])
frame0.insert(1, "category", [tables.category_of_instance[i] for i in scene.frame_track_ids(t)])
frame0["class_id"] = [CLASSES[int(c)] for c in frame0["class_id"]]
frame0["speed_m_s"] = np.hypot(frame0["vx"], frame0["vy"])
print(f"\nKeyframe {t}: the (N_agents, features) matrix, {len(frame0)} agents")
display(frame0.round(2).head(15))

In [ ]:
# Every mini scene: shape and the no-object-loss check.
rows = []
for name in tables.split_scene_names("all"):
    s = build_object_scene(tables, name)
    n_ann = sum(len(tables.annotations_of.get(t, [])) for t in s.sample_tokens)
    per_frame = s.mask.sum(1).double()
    rows.append({
        "scene": name,
        "split": "mini_val" if name in MINI_SPLITS["mini_val"] else "mini_train",
        "tensor [T, N, F]": str(tuple(s.features.shape)),
        "annotations": n_ann,
        "in tensor": s.num_annotations,
        "lossless": s.num_annotations == n_ann,
        "agents / frame (mean)": round(float(per_frame.mean()), 1),
        "agents / frame (max)": int(per_frame.max()),
    })
stats = pd.DataFrame(rows).set_index("scene")
display(stats)
assert stats["lossless"].all()
print(f"All {len(stats)} scenes lossless: {stats['annotations'].sum()} annotations in total.")

In [ ]:
# Who is where and when: presence of every agent over the scene, coloured by class.
CLASS_COLORS = {"vehicle": "#00b4d8", "pedestrian": "#ffaa00", "two_wheeler": "#c77dff", "obstacle": "#8d99ae", "animal": "#80ed99"}
cls = scene.features[..., FEATURES.index("class_id")].long()
first_class = torch.tensor([int(cls[scene.mask[:, n], n][0]) for n in range(scene.mask.shape[1])])
canvas = np.ones(scene.mask.T.shape + (3,))
for k, name in enumerate(CLASSES):
    rgb = np.array([int(CLASS_COLORS[name][i : i + 2], 16) / 255 for i in (1, 3, 5)])
    sel = (first_class == k).numpy()
    canvas[sel] = np.where(scene.mask.T[sel].numpy()[..., None], rgb, 1.0)

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), gridspec_kw={"width_ratios": [2.3, 1]})
axes[0].imshow(canvas, aspect="auto", interpolation="nearest")
axes[0].set_xlabel("keyframe (2 Hz)")
axes[0].set_ylabel("agent (tracking id)")
axes[0].set_title(f"{SCENE}: presence mask [{scene.mask.shape[1]} agents x {scene.mask.shape[0]} keyframes]", fontweight="bold")
counts = pd.Series([CLASSES[int(c)] for c in first_class]).value_counts()
axes[1].barh(counts.index, counts.values, color=[CLASS_COLORS[c] for c in counts.index])
axes[1].set_title("agents per class", fontweight="bold")
for name in CLASSES:
    axes[0].plot([], [], "s", color=CLASS_COLORS[name], label=name)
axes[0].legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "presence_mask.png"), dpi=110)
plt.show()

In [ ]:
# Drawing helpers: the scene from above, in the ego frame of one keyframe.
X, Y, L, W, S, C, VXI, VYI, K = (FEATURES.index(k) for k in ("x", "y", "length", "width", "yaw_sin", "yaw_cos", "vx", "vy", "class_id"))
EGO_L, EGO_W = 4.084, 1.85


def corners(cx, cy, length, width, yaw):
    c, s = math.cos(yaw), math.sin(yaw)
    return [(cx + c * dx - s * dy, cy + s * dx + c * dy) for dx, dy in ((length / 2, width / 2), (length / 2, -width / 2), (-length / 2, -width / 2), (-length / 2, width / 2))]


def lidar_in_ego(token):
    rec = tables.keyframe[(token, "LIDAR_TOP")]
    path = Path(DATA_ROOT) / rec["filename"]
    if not path.exists():
        return None
    pts = np.fromfile(path, dtype=np.float32).reshape(-1, 5)[:, :3].astype(np.float64)
    cal = tables_calib[rec["calibrated_sensor_token"]]
    return pts @ quaternion_to_matrix(cal["rotation"]).T + np.asarray(cal["translation"])


with open(Path(DATA_ROOT) / VERSION / "calibrated_sensor.json") as f:
    tables_calib = {r["token"]: r for r in json.load(f)}


def global_to_ego(xy, token):
    rot, trans = tables.ego_pose(token)
    xy = np.asarray(xy, dtype=np.float64).reshape(-1, 2)
    return (np.c_[xy, np.zeros(len(xy))] - trans) @ rot[:, :2] if len(xy) else xy


def draw_scene(ax, scene_ego, t, view=(-25, 45, -30, 30), label_ids=True):
    token = scene_ego.sample_tokens[t]
    ax.set_facecolor("#11131a")
    pts = lidar_in_ego(token) if HAS_LIDAR else None
    if pts is not None:
        m = (pts[:, 2] > 0.2) & (pts[:, 0] > view[0]) & (pts[:, 0] < view[1]) & (pts[:, 1] > view[2]) & (pts[:, 1] < view[3])
        ax.scatter(pts[m, 0], pts[m, 1], s=0.5, color="#4a5570", alpha=0.6)
    feats = scene_ego.frame_features(t).numpy()
    ids = scene_ego.frame_track_ids(t)
    for f, tid in zip(feats, ids):
        if not (view[0] < f[X] < view[1] and view[2] < f[Y] < view[3]):
            continue
        color = CLASS_COLORS[CLASSES[int(f[K])]]
        ax.add_patch(patches.Polygon(corners(f[X], f[Y], f[L], f[W], math.atan2(f[S], f[C])), closed=True, facecolor=color, edgecolor="white", lw=0.6, alpha=0.85))
        if math.hypot(f[VXI], f[VYI]) > 0.5:
            ax.annotate("", xy=(f[X] + f[VXI], f[Y] + f[VYI]), xytext=(f[X], f[Y]), arrowprops=dict(arrowstyle="->", color="white", lw=1.2))
        if label_ids and abs(f[X]) < 25 and abs(f[Y]) < 15:
            ax.text(f[X], f[Y] + 1.2, tid[:4], color="white", fontsize=6, ha="center")
    ax.add_patch(patches.Polygon(corners(0, 0, EGO_L, EGO_W, 0), closed=True, color="#ff3333", zorder=5))
    ax.annotate("", xy=(4.5, 0), xytext=(0, 0), arrowprops=dict(arrowstyle="-|>", color="white", lw=2), zorder=6)
    ax.set_xlim(view[0], view[1])
    ax.set_ylim(view[2], view[3])
    ax.set_aspect("equal")
    ax.set_xlabel("forward (m)")
    ax.set_ylabel("left (m)")
    ax.grid(True, ls="--", alpha=0.2)

In [ ]:
# One keyframe of the representation: boxes, headings, velocity arrows (= motion in the next 1 s), tracking ids.
fig, ax = plt.subplots(figsize=(10, 8.5))
t = 10
draw_scene(ax, scene, t)
for name in CLASSES:
    ax.plot([], [], "s", color=CLASS_COLORS[name], label=name)
ax.plot([], [], "s", color="#ff3333", label="ego")
ax.legend(loc="upper left", fontsize=8)
ax.set_title(f"{SCENE}, keyframe {t}: {int(scene.mask[t].sum())} agents (white arrow = velocity x 1 s)", fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "object_centric_bev.png"), dpi=110)
plt.show()

In [ ]:
# Object baselines on all mini scenes (no training, so no train/test leakage).
preds = {k: [] for k in BASELINES}
gts, valids = [], []
for name in tables.split_scene_names("all"):
    s = build_object_scene(tables, name, frame="global")
    for _, pos, vel, gt, valid, dt in forecast_windows(s):
        for key, fn in BASELINES.items():
            preds[key].append(fn(pos, vel, dt))
        gts.append(gt)
        valids.append(valid)
gt_all, valid_all = torch.cat(gts), torch.cat(valids)
object_metrics = {key: displacement_errors(torch.cat(p), gt_all, valid_all) for key, p in preds.items()}

rows = []
for key, m in object_metrics.items():
    rows.append({"Baseline": key.replace("_", " ").title(), **{f"ADE @ {h}": m["ade"][h] for h in ("1s", "2s", "3s")},
                 "FDE @ 3s": m["fde"]["3s"], "Miss rate @ 3s (%)": 100 * m["miss_rate"]["3s"]})
print(f"{gt_all.shape[0]} agent windows (agents present at t0 with 6 future keyframes) — metres, lower is better")
display(pd.DataFrame(rows).set_index("Baseline").round(2))

fig, ax = plt.subplots(figsize=(7, 4))
for key, m in object_metrics.items():
    ax.plot([1, 2, 3], [m["fde"][h] for h in ("1s", "2s", "3s")], "o-", label=f"{key} FDE")
    ax.plot([1, 2, 3], [m["ade"][h] for h in ("1s", "2s", "3s")], "s--", alpha=0.7, label=f"{key} ADE")
ax.set_xlabel("horizon (s)")
ax.set_ylabel("error (m)")
ax.set_title("Error growth with the horizon (stability)", fontweight="bold")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Forecasts of every agent from one keyframe: past track, true future, constant velocity, persistence.
scene_global = build_object_scene(tables, SCENE, frame="global")
t0 = 10
win = {w[0]: w for w in forecast_windows(scene_global)}[t0]
_, pos, vel, gt, valid, dt = win
token0 = scene.sample_tokens[t0]
cv = constant_velocity(pos, vel, dt).numpy()
present = torch.nonzero(scene_global.mask[t0]).flatten().tolist()
fig, ax = plt.subplots(figsize=(10, 8.5))
draw_scene(ax, scene, t0, label_ids=False)
for a, n in enumerate(present):
    past = [k for k in range(max(0, t0 - 4), t0 + 1) if scene_global.mask[k, n]]
    past_xy = global_to_ego(scene_global.features[past, n][:, [X, Y]].numpy(), token0)
    fut_xy = global_to_ego(gt[a][valid[a]].numpy(), token0)
    cv_xy = global_to_ego(cv[a], token0)
    if not len(fut_xy) or np.abs(past_xy[-1]).max() > 40:
        continue
    ax.plot(past_xy[:, 0], past_xy[:, 1], ":", color="white", lw=1)
    ax.plot(fut_xy[:, 0], fut_xy[:, 1], "-", color="#ffee00", lw=2)
    ax.plot(cv_xy[:, 0], cv_xy[:, 1], "--", color="#00ff88", lw=1.5)
ax.plot([], [], ":", color="white", label="past (2 s)")
ax.plot([], [], "-", color="#ffee00", label="true future (3 s)")
ax.plot([], [], "--", color="#00ff88", label="constant velocity forecast")
ax.legend(loc="upper left", fontsize=8)
ax.set_title(f"{SCENE}, keyframe {t0}: agent forecasts vs ground truth (persistence = the boxes themselves)", fontweight="bold", fontsize=10)
plt.tight_layout()
plt.savefig(os.path.join(OUT_DIR, "object_forecasts.png"), dpi=110)
plt.show()

In [ ]:
# Animated drive through the scene: agents, velocities and their constant-velocity forecasts.
MAX_FRAMES = 40
frames = []
for t in range(min(MAX_FRAMES, len(scene.sample_tokens))):
    fig, ax = plt.subplots(figsize=(8, 6.8))
    draw_scene(ax, scene, t, label_ids=False)
    feats = scene.frame_features(t).numpy()
    for f in feats:
        if math.hypot(f[VXI], f[VYI]) > 0.5 and abs(f[X]) < 45 and abs(f[Y]) < 30:
            ax.plot([f[X], f[X] + 3 * f[VXI]], [f[Y], f[Y] + 3 * f[VYI]], "--", color="#00ff88", lw=1)
    ax.set_title(f"{SCENE}  keyframe {t + 1}/{len(scene.sample_tokens)}: {len(feats)} agents, dashed = 3 s CV forecast", fontsize=10)
    plt.tight_layout()
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=70)
    plt.close(fig)
    buf.seek(0)
    frames.append(Image.open(buf).convert("P", palette=Image.ADAPTIVE))
gif_path = os.path.join(OUT_DIR, f"{SCENE}_object_centric.gif")
frames[0].save(gif_path, save_all=True, append_images=frames[1:], duration=350, loop=0)
print("Saved", gif_path)
display(IPImage(filename=gif_path, width=700))

In [ ]:
# Ego baselines that read only the past (shared with the BEV representation).
# The midterm constant-velocity baseline took its speed from the first future waypoint; these do not.
ego_metrics = {"all mini scenes": evaluate_ego_baselines(tables, tables.split_scene_names("all"))}
if "scene-0061" in tables.scene_by_name:
    ego_metrics["scene-0061 (BEV notebook scene)"] = evaluate_ego_baselines(tables, ["scene-0061"])
rows = []
for where, res in ego_metrics.items():
    for key, label in (("zero_motion", "Zero-Motion"), ("constant_velocity", "Constant Velocity")):
        rows.append({"scenes": f"{where} ({res['num_windows']} windows)", "Method": label, **{f"L2 @ {h}": res[key][h] for h in ("1s", "2s", "3s")}, "L2 Avg": res[key]["avg"]})
display(pd.DataFrame(rows).set_index(["scenes", "Method"]).round(2))

In [ ]:
# Save everything and print the Markdown rows for README.md / results/experiments.md.
report = {
    "dataset": f"nuScenes {VERSION}",
    "features": list(FEATURES),
    "all_scenes_lossless": bool(stats["lossless"].all()),
    "scenes": stats.reset_index().to_dict(orient="records"),
    "object_baselines": object_metrics,
    "num_agent_windows": int(gt_all.shape[0]),
    "ego_baselines_l2_m": ego_metrics,
}
with open(os.path.join(OUT_DIR, "object_centric_metrics.json"), "w") as f:
    json.dump(report, f, indent=2)
print("Saved", os.path.join(OUT_DIR, "object_centric_metrics.json"), "\n")
print("| Baseline (nuScenes mini, all agents) | ADE @ 1s (m) ↓ | ADE @ 2s (m) ↓ | ADE @ 3s (m) ↓ | FDE @ 3s (m) ↓ | Miss rate @ 3s ↓ |")
print("|---|:---:|:---:|:---:|:---:|:---:|")
for key, m in object_metrics.items():
    print(f"| {key.replace('_', ' ').title()} | {m['ade']['1s']:.2f} | {m['ade']['2s']:.2f} | {m['ade']['3s']:.2f} | {m['fde']['3s']:.2f} | {100 * m['miss_rate']['3s']:.1f}% |")

## Notes
- **No object loss** is checked for every scene above: the number of non-zero mask entries equals the number of annotations.
- **Velocity** is a backward difference, so the representation at keyframe *t* never uses later frames; agents seen for the first time have zero velocity.
- **Metrics:** ADE / FDE / miss rate (> 2 m) per horizon. Constant velocity is the bar a learned object-centric world model has to clear; the error-growth plot is the stability view of the same numbers.
- mini has 10 scenes: treat the numbers as a pipeline check; final results need `v1.0-trainval`.